# Notebook 1: Data Preparation – Illustration

This notebook is an **exploratory illustration** of the dataset structure.
It shows how to load the data from HuggingFace, what the key variables look like,
and how the pre-computed text embeddings can be processed (PCA, clustering).

Notebooks 2 and 3 are fully self-contained and repeat the necessary preprocessing steps inline.

In [ ]:
import datasets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

palette = sns.color_palette("colorblind")

## 1. Load the Dataset

In [ ]:
ds = datasets.load_dataset("janteichertkluge/demand-analysis-repro")
print(ds)

ds_train = ds["train"]
ds_test  = ds["test"]
print(f"\nTrain samples: {len(ds_train):,}")
print(f"Test samples:  {len(ds_test):,}")

In [ ]:
# Identify column groups
all_cols  = ds_train.column_names
emb_cols  = sorted([c for c in all_cols if c.startswith("emb_")])
meta_cols = [c for c in all_cols if c not in emb_cols]

print(f"Embedding dimensions: {len(emb_cols)}")
print(f"Non-embedding columns ({len(meta_cols)}):")
for c in meta_cols:
    print(" ", c)

In [ ]:
df_train = ds_train.to_pandas()
df_test  = ds_test.to_pandas()

df_train = df_train.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})
df_test  = df_test.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})

df_train[["ASIN", "date", "Q_t", "P_bb_t", "subcat_aggregated"]].head()

## 2. Descriptive Statistics

In [ ]:
desc_vars = ["Q_t", "P_bb_t", "REVIEW_COUNT", "RATING"]
desc_vars = [v for v in desc_vars if v in df_train.columns]
df_train[desc_vars].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_train["Q_t"].dropna(), bins=80, color=palette[0], alpha=0.8)
axes[0].set_xlabel("Sales Rank (Q_t)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Quantity (Sales Rank)")

axes[1].hist(df_train["P_bb_t"].dropna(), bins=80, color=palette[1], alpha=0.8)
axes[1].set_xlabel("Buy-Box Price (P_bb_t)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of Price")

plt.tight_layout()
plt.show()

In [ ]:
if "subcat_aggregated" in df_train.columns:
    print("Products per subcategory (train):")
    print(df_train["subcat_aggregated"].value_counts().to_string())

## 3. Embedding Processing

We center and L2-normalize the embeddings, then visualize them via PCA.

In [ ]:
emb_train = df_train[emb_cols].values
emb_test  = df_test[emb_cols].values
emb_all   = np.vstack([emb_train, emb_test])

# Center and normalize
emb_centered = emb_all - emb_all.mean(axis=0)
emb_normed   = normalize(emb_centered, axis=1)
print(f"Embedding matrix: {emb_normed.shape}")

In [ ]:
pca = PCA(n_components=10, random_state=42)
pca.fit(emb_normed[:len(df_train)])

print("Explained variance per component:")
for i, ev in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {ev:.3f}")
print(f"  Cumulative (10 PCs): {pca.explained_variance_ratio_.cumsum()[-1]:.3f}")

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(emb_normed[:len(df_train)])

# Color by subcategory if available
if "subcat_aggregated" in df_train.columns:
    cats = df_train["subcat_aggregated"].astype("category")
    codes = cats.cat.codes.values
else:
    codes = np.zeros(len(coords), dtype=int)
    cats  = pd.Categorical(["all"] * len(coords))

fig, ax = plt.subplots(figsize=(8, 6))
for i, cat in enumerate(cats.cat.categories):
    mask = codes == i
    ax.scatter(coords[mask, 0], coords[mask, 1],
               label=cat, alpha=0.3, s=6, color=palette[i % len(palette)])
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("Text Embeddings – First Two Principal Components (train)")
ax.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.show()

## 4. K-Means Cluster Structure

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=50)
labels = kmeans.fit_predict(emb_normed[:len(df_train)])

fig, ax = plt.subplots(figsize=(8, 6))
for k in range(5):
    mask = labels == k
    ax.scatter(coords[mask, 0], coords[mask, 1],
               label=f"Cluster {k}", alpha=0.3, s=6, color=palette[k])
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("K-Means Clusters in Embedding Space (train)")
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

## 5. Time Series Structure

In [ ]:
if "date" in df_train.columns:
    df_train["date"] = pd.to_datetime(df_train["date"])
    monthly = df_train.groupby("date")[["Q_t", "P_bb_t"]].median()

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    axes[0].plot(monthly.index, monthly["Q_t"],   color=palette[0])
    axes[0].set_ylabel("Median Sales Rank")
    axes[0].set_title("Median Sales Rank over Time (train)")
    axes[0].grid(alpha=0.3)

    axes[1].plot(monthly.index, monthly["P_bb_t"], color=palette[1])
    axes[1].set_ylabel("Median Price ($)")
    axes[1].set_title("Median Buy-Box Price over Time (train)")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()